# Evaluasi pada Data Sangat Tidak Seimbang: Deteksi Penipuan Kartu Kredit

**Buku:** *Deep Learning dengan Python* — Bab 16, Mengevaluasi Model dengan Jujur

Hanya 0,17% transaksi yang merupakan penipuan. Notebook ini menunjukkan mengapa akurasi menyesatkan, bagaimana membaca kurva ROC dan presisi–recall, cara memilih ambang keputusan yang benar, serta cara melaporkan ketidakpastian.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±5 menit

**Isi notebook:**
- Paradoks akurasi: model "selalu negatif" mencapai akurasi 99,8%.
- Tiga strategi pelatihan: polos, `class_weight`, dan inisialisasi bias keluaran.
- ROC-AUC vs PR-AUC (*average precision*), dan mengapa PR-AUC lebih jujur di sini.
- Memilih ambang di data **validasi** (F1 maksimum atau target recall), lalu menerapkannya sekali di data uji.
- Kalibrasi (diagram reliabilitas, Brier score) dan selang kepercayaan *bootstrap*.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Data

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv")
neg, pos = np.bincount(df["Class"])
print(f"{len(df):,} transaksi | positif (penipuan): {pos} ({pos / len(df):.3%})")

df = df.drop(columns=["Time"])
df["Amount"] = np.log1p(df["Amount"])
y = df.pop("Class").values
X = df.values.astype("float32")

# Pembagian berstrata 60/20/20 agar proporsi positif sama di semua bagian
X_latih, X_sisa, y_latih, y_sisa = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_uji, y_val, y_uji = train_test_split(X_sisa, y_sisa, test_size=0.5, stratify=y_sisa, random_state=42)
skaler = StandardScaler().fit(X_latih)                      # fit HANYA pada data latih
X_latih, X_val, X_uji = (np.clip(skaler.transform(a), -5, 5) for a in (X_latih, X_val, X_uji))
print("Positif di latih/val/uji:", y_latih.sum(), y_val.sum(), y_uji.sum())

## 2. Paradoks akurasi

In [ ]:
print(f"Akurasi model yang SELALU menebak 'bukan penipuan': {(y_uji == 0).mean():.3%}")
print("Recall-nya: 0%. Model ini tidak berguna, tetapi akurasinya nyaris sempurna.")

## 3. Tiga strategi pelatihan

In [ ]:
from keras import layers

def metrik():          # objek metrik baru untuk setiap model (jangan dipakai bersama)
    return [keras.metrics.Precision(name="presisi"), keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="roc_auc"), keras.metrics.AUC(curve="PR", name="pr_auc")]

def bangun(bias_awal=None):
    inisial = keras.initializers.Constant(bias_awal) if bias_awal is not None else "zeros"
    model = keras.Sequential([keras.Input(shape=(X_latih.shape[1],)),
                              layers.Dense(32, activation="relu"), layers.Dropout(0.3),
                              layers.Dense(1, activation="sigmoid", bias_initializer=inisial)])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=metrik())
    return model

bias_awal = np.log(y_latih.sum() / (len(y_latih) - y_latih.sum()))     # log(pos/neg)
bobot_kelas = {0: len(y_latih) / (2 * (len(y_latih) - y_latih.sum())), 1: len(y_latih) / (2 * y_latih.sum())}
print(f"Bias awal: {bias_awal:.2f} (prob awal = {1 / (1 + np.exp(-bias_awal)):.4f}) | bobot kelas: {bobot_kelas}")

strategi = {"polos": (bangun(), None),
            "class_weight": (bangun(), bobot_kelas),
            "bias_awal": (bangun(bias_awal), None)}
hentikan = keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=5, restore_best_weights=True)
prob_val, prob_uji = {}, {}
for nama, (m, bobot) in strategi.items():
    m.fit(X_latih, y_latih, batch_size=2048, epochs=50, validation_data=(X_val, y_val),
          class_weight=bobot, callbacks=[hentikan], verbose=0)
    prob_val[nama] = m.predict(X_val, batch_size=8192, verbose=0).ravel()
    prob_uji[nama] = m.predict(X_uji, batch_size=8192, verbose=0).ravel()
    print(f"{nama:13s} selesai")

## 4. ROC vs presisi–recall
Pada ROC, laju positif palsu (FPR) dibagi jumlah negatif yang sangat besar, sehingga ratusan alarm palsu tetap tampak kecil. Kurva PR tidak bisa "bersembunyi" seperti itu. Garis dasar PR-AUC = prevalensi (0,0017), bukan 0,5.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for nama, p in prob_uji.items():
    fpr, tpr, _ = roc_curve(y_uji, p)
    pre, rec, _ = precision_recall_curve(y_uji, p)
    ax[0].plot(fpr, tpr, label=f"{nama} (AUC {roc_auc_score(y_uji, p):.3f})")
    ax[1].plot(rec, pre, label=f"{nama} (AP {average_precision_score(y_uji, p):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set(xlabel="FPR", ylabel="TPR (recall)", title="ROC")
ax[1].axhline(y_uji.mean(), color="k", ls="--", lw=1, label="acak = prevalensi")
ax[1].set(xlabel="Recall", ylabel="Presisi", title="Presisi–Recall")
for a in ax: a.legend(fontsize=9)
plt.show()

## 5. Memilih ambang keputusan
Ambang 0,5 bukan hukum alam. Ambang dipilih di data **validasi** sesuai kebutuhan bisnis, lalu dibekukan sebelum menyentuh data uji.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, ConfusionMatrixDisplay

def ambang_f1_maks(y, p):
    pre, rec, ambang = precision_recall_curve(y, p)
    f1 = 2 * pre * rec / np.clip(pre + rec, 1e-12, None)
    return ambang[np.argmax(f1[:-1])]

def ambang_target_recall(y, p, target=0.85):
    pre, rec, ambang = precision_recall_curve(y, p)
    ok = np.where(rec[:-1] >= target)[0]
    return ambang[ok[np.argmax(pre[ok])]]            # presisi terbaik yang masih memenuhi target recall

NAMA = "bias_awal"
p_val, p_uji = prob_val[NAMA], prob_uji[NAMA]
pilihan = {"0,5 (default)": 0.5,
           "F1 maks (val)": ambang_f1_maks(y_val, p_val),
           "recall ≥ 0,85 (val)": ambang_target_recall(y_val, p_val)}
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, (label, th) in zip(ax, pilihan.items()):
    pred = (p_uji >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_uji, pred).ravel()
    ConfusionMatrixDisplay(confusion_matrix(y_uji, pred), display_labels=["sah", "penipuan"]).plot(ax=a, colorbar=False)
    a.set_title(f"ambang {label} = {th:.3f}\npresisi {tp / max(tp + fp, 1):.2f} | recall {tp / (tp + fn):.2f} | F1 {f1_score(y_uji, pred):.2f}")
plt.tight_layout(); plt.show()

## 6. Kalibrasi
`class_weight` biasanya menaikkan recall, tetapi probabilitasnya menjadi **terlalu tinggi** (tidak terkalibrasi). Ini penting bila skor dipakai sebagai peluang, misalnya untuk menghitung ekspektasi kerugian.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

plt.figure(figsize=(6, 5))
for nama, p in prob_uji.items():
    benar, prediksi = calibration_curve(y_uji, p, n_bins=10, strategy="quantile")
    plt.plot(prediksi, benar, "o-", label=f"{nama} (Brier {brier_score_loss(y_uji, p):.5f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Probabilitas prediksi rata-rata"); plt.ylabel("Frekuensi positif sebenarnya")
plt.title("Diagram reliabilitas (skala log)"); plt.legend(fontsize=9); plt.show()

## 7. Ketidakpastian: selang kepercayaan *bootstrap*
Data uji hanya berisi ±98 positif, sehingga satu angka PR-AUC tanpa selang kepercayaan bisa menipu.

In [ ]:
def bootstrap_ci(y, p, fungsi, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    nilai = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() > 0:
            nilai.append(fungsi(y[i], p[i]))
    return np.percentile(nilai, [2.5, 97.5])

for nama, p in prob_uji.items():
    ap = average_precision_score(y_uji, p)
    lo, hi = bootstrap_ci(y_uji, p, average_precision_score)
    print(f"{nama:13s} PR-AUC = {ap:.3f}  (IK 95%: {lo:.3f} – {hi:.3f})")

Bila selang kepercayaan dua strategi saling tumpang tindih lebar, kita **belum** punya bukti bahwa yang satu lebih baik. Untuk perbandingan berpasangan yang lebih tegas, *bootstrap* selisihnya pada sampel ulang yang sama, atau pakai uji DeLong untuk ROC-AUC.